In [ ]:

import parmed
from sklearn.cluster import DBSCAN
from MDAnalysis.analysis import contacts
from MDAnalysis.analysis import distances
from collections import Counter
import nglview as nv
import MDAnalysis as mda
import numpy as np
from MDAnalysis.transformations import wrap, center_in_box
from scipy.spatial.distance import pdist,squareform
import seaborn as sns
import pytim
import warnings
import pandas as pd


In [ ]:
### For RNA G and A, it is 13
### For RNA U and C, it is 11.5

def clustering(u, frame,eps=13):

        ### Function that is nested in all of the analysis methods in this file ###

        ca_sel = u.select_atoms("name C1' or name CA")

        cas = [atom.index for atom in ca_sel.atoms]

        to_cluster = [atom for i, atom in enumerate(u.trajectory[frame.frame].positions)if i in cas]

        n_ca = len(ca_sel)

        self_distances = distances.self_distance_array(ca_sel.positions,
                                                       box=frame.dimensions)

        sq_dist_arr = np.zeros((n_ca, n_ca))
        triu = np.triu_indices_from(sq_dist_arr, k=1)

        sq_dist_arr[triu] = self_distances
        sq_dist_arr.T[triu] = self_distances
  
        model = DBSCAN(eps=eps, min_samples=1, metric='precomputed')
    
        labels = model.fit_predict(sq_dist_arr)
        b = Counter(labels)

        clustered_list = [atom for j, atom in enumerate(to_cluster)
                          for i, item in enumerate(labels)
                          if item == b.most_common()[0][0]
                          and j == i]

        centroid = np.mean(clustered_list, axis=0)

        ind_list = [cas[index] for index, j in enumerate(to_cluster)
                    for i in clustered_list
                    if j[0]==i[0] and j[1]==i[1] and j[2]==i[2]]

        index_list = [x+1 for x in ind_list]

        res_list = [u.atoms[i].resindex for i in index_list]

        return centroid, to_cluster, clustered_list, res_list
    
    
    
def surface_comp2(u, start=4000, stop=-1, step=10000):
    frame_num = 0
    inner_layers_all = []
    outer_layers_all = []

    for frame in u.trajectory[start:stop:step]:
        
        centroid, to_cluster, clustered_list, res_list = clustering(u, frame, eps=9)

        # --- Build selection for clustered core residues ---
        selection = 'resindex '
        for item in res_list:
            selection += str(item) + ' or resindex '
        selection = selection[0:-13]

        sel = u.select_atoms(selection)

        # --- Write clustered residues to temp PDB and reload ---
        with mda.Writer('spatial.pdb') as pdb:
            pdb.write(sel)
        univ = mda.Universe('spatial.pdb')

        # --- Run GITIM surface analysis ---
        g = univ.select_atoms('name CA')
        interface = pytim.GITIM(
            univ,
            group=g,
            alpha=4.5,
            max_layers=3,
            molecular=False,
            centered='middle'
        )

        # --- Extract layer 1 (outer) and layer 2 (inner) via interface object ---
        layer1_atoms = interface.layers[0]   # outer surface
        layer2_atoms = interface.layers[1]   # mid layer
        layer3_atoms = interface.layers[2]   # inner layer

        outer_resids = np.unique(layer1_atoms.resids)
        mid_resids = np.unique(layer2_atoms.resids)
        inner_resids = np.unique(layer3_atoms.resids)
        
        
        return inner_resids ,mid_resids,outer_resids,res_list
    

def surface_comp3(u, start=4000, stop=-1, step=10000):
    
    frame_num = 0
    inner_layers_all = []
    outer_layers_all = []

    for frame in u.trajectory[start:stop:step]:
        centroid, to_cluster, clustered_list, res_list = clustering(u, frame, eps=9)

        # --- Build selection for clustered core residues ---
        selection = 'resindex '
        for item in res_list:
            selection += str(item) + ' or resindex '
        selection = selection[0:-13]

        sel = u.select_atoms(selection)

        # --- Write clustered residues to temp PDB and reload ---
        with mda.Writer('spatial.pdb') as pdb:
            pdb.write(sel)
        univ = mda.Universe('spatial.pdb')

        # --- Run GITIM surface analysis ---
        g = univ.select_atoms('name CA')
        interface = pytim.GITIM(
            univ,
            group=g,
            alpha=4.2,
            max_layers=2,
            molecular=False,
            centered='middle'
        )

        # --- Extract layer 1 (outer) and layer 2 (inner) via interface object ---
        layer1_atoms = interface.layers[0]   # outer surface
        layer2_atoms = interface.layers[1]   # mid layer
        #layer3_atoms = interface.layers[2]   # inner layer

        outer_resids = np.unique(layer1_atoms.resids)
        inner_resids = np.unique(layer2_atoms.resids)
        
        return inner_resids,outer_resids,res_list


def surface_comp_new(u):
    
    inner_layers_all = []
    outer_layers_all = []
    frame = u.trajectory[0]
    
    centroid, to_cluster, clustered_list, res_list = clustering(u,frame, eps=9)

    # --- Build selection for clustered core residues ---
    selection = 'resindex '
    for item in res_list:
        selection += str(item) + ' or resindex '
    selection = selection[0:-13]

    sel = u.select_atoms(selection)

    # --- Write clustered residues to temp PDB and reload ---
    with mda.Writer('spatial.pdb') as pdb:
        pdb.write(sel)
    univ = mda.Universe('spatial.pdb')

    # --- Run GITIM surface analysis ---
    g = univ.select_atoms('name CA')
    interface = pytim.GITIM(
        univ,
        group=g,
        alpha=3.9,
        max_layers=2,
        molecular=False,
        centered='middle'
    )

    # --- Extract layer 1 (outer) and layer 2 (inner) via interface object ---
    layer1_atoms = interface.layers[0]   # outer surface
    layer2_atoms = interface.layers[1]   # mid layer
    #layer3_atoms = interface.layers[2]   # inner layer

    outer_resids = np.unique(layer1_atoms.resids)
    #inner_resids = np.unique(layer2_atoms.resids)
     
    inner_resids = np.setdiff1d(res_list, outer_resids)

    print(len(outer_resids), len(inner_resids), len(res_list))

    return inner_resids, outer_resids, res_list




def visu(u):
    view=nv.show_mdanalysis(u)
    view.clear_representations()
    view.add_representation('ball+stick',sele='protein',aspectRatio=2)
   # view.add_representation('surface',sele='LYS',opacity=0.8,color='green')
    #view.add_representation('ball+stick',sele='LYS',aspectRatio=2)
    view.add_representation('ball+stick',sele='ions',aspectRatio=3)
    #view.add_representation('ball+stick',sele='water',aspectRatio=2)
    view.add_unitcell(color='white')
    return view

def visu2(u):
    view=nv.show_mdanalysis(u)
    view.clear_representations()
    view.add_representation('ball+stick',sele='protein',aspectRatio=5)
    
    #view.add_representation('surface',sele='protein',opacity=0.3,color='silver')

   # view.add_representation('surface',sele='LYS',opacity=0.8,color='green')
    #view.add_representation('ball+stick',sele='LYS',aspectRatio=2)
    view.add_representation('ball+stick',sele='ions',aspectRatio=3)
    #view.add_representation('ball+stick',sele='water',aspectRatio=2)
    view.add_unitcell(color='white')
    return view

def visu3(u):
    view=nv.show_mdanalysis(u)
    view.clear_representations()
    view.add_representation('surface',sele='protein',opacity=0.8,color='green')
    view.add_representation('ball+stick',sele='nucleic',aspectRatio=2)
    view.add_representation('surface',sele='LYS',opacity=0.8,color='green')
    #view.add_representation('ball+stick',sele='LYS',aspectRatio=2)
    view.add_representation('surface',sele='nucleic',opacity=0.8)
    view.add_representation('ball+stick',sele='ions',aspectRatio=3)
    #view.add_representation('ball+stick',sele='water',aspectRatio=2)
    view.add_unitcell(color='white')
    return view


def visul5(unive,s1,s2,water=False):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=5,color='red')  
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=1.5,aspectRatio=3.5,color='yellow')  
    
    view.add_representation('ball+stick',sele=f"(nucleic or protein ) and ({s2})",
                            opacity=1.5,
                            aspectRatio=3.5,
                            color='green')   
    if water==True:
    
        view.add_representation('ball+stick',sele=f"(SOL )",
                                opacity=0.2,
                                aspectRatio=2,
                                color='silver')
    #view.add_unitcell(color='white')
    return view

def visul5_new(unive,water=False):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic))",
                            opacity=1.5,aspectRatio=5,color='red')  
    
    view.add_representation('ball+stick',sele=f"(protein))",
                            opacity=1.5,aspectRatio=3.5,color='yellow')  
    
    
    view.add_representation('ball+stick',sele=f"(SOL )",
                            opacity=0.2,
                            aspectRatio=2,
                            color='silver')
    if water==True:
    
        view.add_representation('ball+stick',sele=f"(SOL )",
                                opacity=0.2,
                                aspectRatio=2,
                                color='silver')
#    view.add_unitcell(color='white')

    return view


def visul6(unive,s1,s2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='red')  
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(SOL ) and ({s2})",
                            opacity=1.5,
                            aspectRatio=8,
                            color='green')
    
    

    view.add_unitcell(color='white')
    return view


def visul6_(unive,s1,s2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='red')  
    
#     view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
#                             opacity=1.5,aspectRatio=2)  
    
    view.add_representation('ball+stick',sele=f"(SOL ) and ({s2})",
                            opacity=1.5,
                            aspectRatio=4.2
                           )
    
    view.add_representation('surface',sele=f"(protein) and ({s1})",
                            opacity=1.5,
                            type='dot', isolevel=20,
                            side='front', depthWrite=False,
                            color='grey')

    
#    view.add_unitcell(color='white')
    return view

def visul6__(unive,s1,s2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='red')  
    
#     view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
#                             opacity=1.5,aspectRatio=2)  
    
    view.add_representation('ball+stick',sele=f"(SOL ) and ({s2})",
                            opacity=1.5,
                            aspectRatio=4.2
                           )
    
    view.add_representation('surface',sele=f"(protein) and ({s1})",
                            opacity=0.18,
                            type='dot', isolevel=20,
                            side='front', depthWrite=False,
                            color='grey')

    
#    view.add_unitcell(color='white')
    return view

def visul7(unive,s1,s2,s3):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='red')  
    
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=1.5,aspectRatio=4,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(protein) and ({s2})",
                            opacity=1.5,
                            aspectRatio=4,
                            color='green')

    view.add_representation('ball+stick',sele=f"(protein) and ({s3})",
                            opacity=1.5,
                            aspectRatio=4,
                            color='red')
    
    

    view.add_unitcell(color='white')
    return view


def visul8(unive,s1,s2,s3,water):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=0.5,aspectRatio=2,color='red')  
    
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=0.5,aspectRatio=4,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(protein) and ({s2})",
                            opacity=0.5,
                            aspectRatio=4,
                            color='green')

    view.add_representation('ball+stick',sele=f"(protein) and ({s3})",
                            opacity=0.5,
                            aspectRatio=4,
                            color='red')
    
    view.add_representation('ball+stick',sele=f"{water}",
                            opacity=1.5,
                            aspectRatio=4,
                            color='orange')
        

    view.add_unitcell(color='white')
    return view


def visul9(unive,s1,s2,s3,water):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('surface',sele=f"(nucleic) and ( {s1})",
                            opacity=0.5,aspectRatio=2,color='red')  
    
    
    view.add_representation('surface',sele=f"(protein) and ( {s1})",
                            opacity=0.5,aspectRatio=4,color='yellow')  
    

    view.add_representation('surface',sele=f"(protein) and ({s2})",
                            opacity=0.5,
                            aspectRatio=4,
                            color='green')

    view.add_representation('surface',sele=f"(protein) and ({s3})",
                            opacity=0.5,
                            aspectRatio=4,
                            color='red')
    
    view.add_representation('ball+stick',sele=f"{water}",
                            opacity=1.5,
                            aspectRatio=4,
                            color='orange')
        

    view.add_unitcell(color='white')
    return view



def visul10(unive,s1,s2,water):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=0.5,aspectRatio=2,color='red')  
    
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=0.5,aspectRatio=4,
                            color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(protein) and ({s2})",
                            opacity=0.5,
                            aspectRatio=4,
                            color='green')

    view.add_representation('ball+stick',sele=f"{water}",
                            opacity=1.5,
                            aspectRatio=6,
                            color='blue')

#    view.add_unitcell(color='white')
    return view

def visul10_water(unive,s1,s2,water):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=0.5,aspectRatio=2,color='red')  
    
    
    view.add_representation('ball+stick',sele=f"(protein and not GLY) and ( {s1})",
                            opacity=0.9,aspectRatio=3,
                            #color='yellow'
                           )  
    

    view.add_representation('ball+stick',sele=f"(protein) and ({s2})",
                            opacity=0.9,
                            aspectRatio=4,
                            color='green')

    view.add_representation('ball+stick',sele=f"{water}",
                            opacity=1.5,
                            aspectRatio=6,
                            #color='blue'
                           )
        

#    view.add_unitcell(color='white')
    return view


def visul11(unive,s1,s2,water1,water2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
#     view.add_representation('surface',sele=f"(nucleic) and ( {s1})",
#                             opacity=0.5,aspectRatio=2,color='red')  
    
    
#     view.add_representation('surface',sele=f"(protein) and ( {s1})",
#                             opacity=0.5,aspectRatio=4,color='yellow')  
    

#     view.add_representation('surface',sele=f"(protein) and ({s2})",
#                             opacity=0.5,
#                             aspectRatio=4,
#                             color='green')

#     view.add_representation('surface',sele=f"(protein) and ({s3})",
#                             opacity=0.5,
#                             aspectRatio=4,
#                             color='red')

    view.add_representation('ball+stick',sele=f"{water1}",
                            opacity=1.5,
                            aspectRatio=6,
                            color='blue')

    
    view.add_representation('ball+stick',sele=f"{water2}",
                            opacity=1.5,
                            aspectRatio=6,
                            color='red')
        

    view.add_unitcell(color='white')
    return view


def visul12(unive,s1,s2,water1,water2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
#     view.add_representation('surface',sele=f"(nucleic) and ( {s1})",
#                             opacity=0.5,aspectRatio=2,color='red')  
    
    
    view.add_representation('surface',sele=f"(protein) and ( {s1})",
                            opacity=0.43,aspectRatio=4,
                            type='dot', isolevel=20,
                            side='front', depthWrite=False,
                            color='yellow')  
    

    view.add_representation('surface',sele=f"(protein) and ({s2})",
                            opacity=0.43,
                            aspectRatio=4,
                            type='dot', isolevel=20,
                            side='front', depthWrite=False,
                            color='green')

#     view.add_representation('surface',sele=f"(protein) and ({s3})",
#                             opacity=0.5,
#                             aspectRatio=4,
#                             color='red')

    view.add_representation('ball+stick',sele=f"{water1}",
                            opacity=1.5,
                            aspectRatio=6,
                            color='blue')

    
    view.add_representation('ball+stick',sele=f"{water2}",
                            opacity=1.5,
                            aspectRatio=6,
                            color='red')
        

    #view.add_unitcell(color='white')
    return view


def visul13(unive,s1,s2,water1,water2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
#     view.add_representation('surface',sele=f"(nucleic) and ( {s1})",
#                             opacity=0.5,aspectRatio=2,color='red')  
    
    
#     view.add_representation('surface',sele=f"(protein) and ( {s1})",
#                             opacity=0.5,aspectRatio=4,color='yellow')  
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=0.4,aspectRatio=5,
                            color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(protein) and ({s2})",
                            opacity=0.4,
                            aspectRatio=5,
                            color='green')

    view.add_representation('ball+stick',sele=f"{water1}",
                            opacity=1.5,
                            aspectRatio=6,
                            color='blue')

    
    view.add_representation('ball+stick',sele=f"{water2}",
                            opacity=1.5,
                            aspectRatio=6,
                            color='red')
        

    #view.add_unitcell(color='white')
    return view

In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/charged//output.top',
                       xyz='../gxg_comps/9_res/charged/output.gro')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 #'../gxg_comps/9_res/charged/traj.dcd',
                 '../gxg_comps/9_res/charged/traj-nw.dcd'
                )
univ.trajectory


In [ ]:
visu(univ)

### Traj 

In [ ]:

stru = parmed.load_file('../gxg_comps/3_res/charged///output.top',
                       xyz='../gxg_comps/3_res/charged//output.gro')


In [ ]:

univ = mda.Universe(stru,
                 '../gxg_comps/3_res/charged//trajectory.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory


In [ ]:

stru = parmed.load_file('../gxg_comps/3_res/nonpolar///output.top',
                       xyz='../gxg_comps/3_res/nonpolar//output.gro')


In [ ]:

univ = mda.Universe(stru,
                 '../gxg_comps/3_res/nonpolar//trajectory.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory


In [ ]:

stru = parmed.load_file('../gxg_comps/3_res/polar///output.top',
                       xyz='../gxg_comps/3_res/polar//output.gro')


univ = mda.Universe(stru,
                 '../gxg_comps/3_res/polar//trajectory.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory

In [ ]:

stru = parmed.load_file('../gxg_comps/3_res/equal///output.top',
                       xyz='../gxg_comps/3_res/equal///output.gro')

univ = mda.Universe(stru,
                 '../gxg_comps/3_res/equal//trajectory.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory

In [ ]:

stru = parmed.load_file('../gxg_comps/3_res/aromatic/output.top',
                       xyz='../gxg_comps/3_res/aromatic/output.gro')

univ = mda.Universe(stru,
                 '../gxg_comps/3_res/aromatic/trajectory.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory

In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/charged///output.top',
                       xyz='../gxg_comps/9_res/charged//output.gro')


In [ ]:
univ = mda.Universe(stru,
                 '../gxg_comps/9_res/charged/traj.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory

In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/nonpolar///output.top',
                       xyz='../gxg_comps/9_res/nonpolar///output.gro')


In [ ]:
univ = mda.Universe(stru,
                 '../gxg_comps/9_res/nonpolar//traj.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory

In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/polar///output.top',
                       xyz='../gxg_comps/9_res/polar/output.gro')


In [ ]:

univ = mda.Universe(stru,
                 '../gxg_comps/9_res/polar//traj.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory

In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/equal///output.top',
                       xyz='../gxg_comps/9_res/equal//output.gro')


In [ ]:
univ = mda.Universe(stru,
                 '../gxg_comps/9_res/equal/traj.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory

In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/aromatic//output.top',
                       xyz='../gxg_comps/9_res/aromatic/output.gro')


In [ ]:
univ = mda.Universe(stru,
                 '../gxg_comps/9_res/aromatic/traj.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory

In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/equal/2_close/output.top',
                       xyz='../gxg_comps/9_res/equal/2_close/output.gro')


In [ ]:
univ = mda.Universe(stru,
                 '../gxg_comps/9_res/equal/2_close/traj.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory

In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/equal/2_spread//output.top',
                       xyz='../gxg_comps/9_res/equal/2_spread//output.gro')


In [ ]:

univ = mda.Universe(stru,
                 '../gxg_comps/9_res/equal/2_spread/traj.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory


In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/equal/3_spread///output.top',
                       xyz='../gxg_comps/9_res/equal/3_spread//output.gro')


In [ ]:

univ = mda.Universe(stru,
                 '../gxg_comps/9_res/equal/3_spread/traj.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory

In [ ]:

view = visul5_new(univ)
view


In [ ]:
frame_id = 67000
frame_id = 62000
#frame_id = 115882
frame_id = 58077
frame_id = 62816
frame_id = 124999
# Move to that frame
for ts in univ.trajectory[frame_id:frame_id+1]:
# Save coordinates of that frame
    univ.atoms.write(f"frame_{frame_id}.pdb")


In [ ]:

#frame_id = 115882
uni = mda.Universe(stru,
                 f'frame_{frame_id}.pdb',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )


In [ ]:

view = visu(uni)
view


In [ ]:

u = uni

for ts in u.trajectory:

    centroid, to_cluster, clustered_list, res_list = clustering(
        u, ts, eps=9
    )
         
    selection = 'resindex '
    for item in res_list:
            selection += str(item)+ ' or resindex '
    selection = selection[0:-13]

    res_list = np.array(res_list)

    selection_2 = ' '
    for item in res_list:
            selection_2 += str(item)+ ' '       

    sel = np.unique(univ.select_atoms(f"(nucleic or protein) and not ({selection})").resids)

    selection_3 = ' '
    for item in sel:
            selection_3 += str(item)+ ' '


In [ ]:

solute = u.select_atoms(f"{selection}")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(mda.transformations.unwrap(u.select_atoms("all") ),
    center_in_box(solute),
    wrap(u.select_atoms("all"), compound='fragments')
)


In [ ]:
name = '3_aromatic'
#u.atoms.write(f"frame2_{frame_id}.pdb")
u.atoms.write(f"frame2_{name}.pdb")


In [ ]:
name = '3_close'
uni = mda.Universe(
                #stru,
                 f'frame2_{name}.pdb',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )


In [ ]:

view = visul5(uni,selection_2,selection_3,
              water=True)
view


In [ ]:

view.render_image(factor=5,
                  transparent=True,
                 trim=True)
#view.download_image(filename='Figures/output.png')

### total water around the droplet

In [ ]:

stru = parmed.load_file('../gxg_comps/3_res/charged///output.top',
                       xyz='../gxg_comps/3_res/charged//output.gro')


In [ ]:
# or explicitly by atom names:
#stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 '../gxg_comps/3_res/charged//trajectory.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory


In [ ]:

stru = parmed.load_file('../gxg_comps/3_res/nonpolar///output.top',
                       xyz='../gxg_comps/3_res/nonpolar//output.gro')


In [ ]:

univ = mda.Universe(stru,
                 '../gxg_comps/3_res/nonpolar//trajectory.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory


In [ ]:

view = visu(univ)
view

In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/charged///output.top',
                       xyz='../gxg_comps/9_res/charged//output.gro')



In [ ]:
# or explicitly by atom names:
#stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 '../gxg_comps/9_res/charged/traj.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory


In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/nonpolar///output.top',
                       xyz='../gxg_comps/9_res/nonpolar//output.gro')



In [ ]:

univ = mda.Universe(stru,
                 '../gxg_comps/9_res/nonpolar//traj.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory


In [ ]:

view = visu(univ)
view


In [ ]:

frame_id = 124000
frame_id = 104679
# frame_id = 67000
# frame_id = 115853
# Move to that frame
for ts in univ.trajectory[frame_id:frame_id+1]:
# Save coordinates of that frame
    univ.atoms.write(f"frame_{frame_id}.pdb")


In [ ]:
frame_id = 67000
uni = mda.Universe(stru,
                 f'frame_{frame_id}.pdb',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )


In [ ]:

view = visu(uni)
view


In [ ]:
## centered
name = '3_charge'
uni = mda.Universe(stru,
                 f'frame2_{name}.pdb',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

In [ ]:
u = uni

for ts in u.trajectory:

    centroid, to_cluster, clustered_list, res_list = clustering(
        u, ts, eps=9
    )
         
    selection = 'resindex '
    for item in res_list:
            selection += str(item)+ ' or resindex '
    selection = selection[0:-13]

    res_list = np.array(res_list)

    selection_2 = ' '
    for item in res_list:
            selection_2 += str(item)+ ' '       

    sel = np.unique(univ.select_atoms(f"(nucleic or protein) and not ({selection})").resids)

    selection_3 = ' '
    for item in sel:
            selection_3 += str(item)+ ' '


In [ ]:

solute = u.select_atoms(f"{selection}")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(mda.transformations.unwrap(u.select_atoms("all") ),
    center_in_box(solute),
    wrap(u.select_atoms("all"), compound='fragments')
)


In [ ]:

view = visul5(u,selection_2,selection_3)
view


In [ ]:

u.atoms.write(f"frame2_{frame_id}.pdb")


In [ ]:

dist_water = 3.0
#u.trajectory[67000]
# Step 2: Water within dist_water of the CORE (clustered) residues
select = f" name OW HW1 HW2 and (around {dist_water} (protein and resid {selection_2}))"
water_near_core = u.select_atoms(select)

water_resids = water_near_core.indices

idx_sel = "@" + ",".join(str(r) for r in water_resids)


In [ ]:

view = visul6_(u,selection_2,idx_sel)
view


In [ ]:

view.render_image(factor=10,
                  transparent=True,
                 trim=True)


In [ ]:

# --- Compute center of condensate ---
com = u.select_atoms(f"{selection}").atoms.center_of_mass()

selection = u.select_atoms(f"prop z > {com[2]}")

# Write cross section
selection.write(f"cross_section_{name}.pdb")

print("Number of atoms in slice:", len(selection))

In [ ]:
# u2 = mda.Universe(f'cross_section_{frame_id}.pdb',
#                  #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
#                 )

u2 = mda.Universe(f'cross_section_{name}.pdb',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

In [ ]:

u = mda.Universe(stru,
                 f'frame2_{frame_id}.pdb',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )


In [ ]:

dist_water = 3.0
#u.trajectory[67000]
# Step 2: Water within dist_water of the CORE (clustered) residues
select = f" name OW HW1 HW2 and (around {dist_water} (protein and resid {selection_2}))"
water_near_core = u2.select_atoms(select)

water_resids = water_near_core.indices

idx_sel = "@" + ",".join(str(r) for r in water_resids)


In [ ]:

view = visul6__(u2,selection_2,idx_sel)
view

## Inner and outer 

In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/charged///output.top',
                       xyz='../gxg_comps/9_res/charged//output.gro')


In [ ]:

#frame_id = 124000
u = mda.Universe(stru,
                 #f'frame2_{frame_id}.pdb',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                 'frame2_9_charge.pdb'
                )


In [ ]:
u = mda.Universe(
                 f'spatial.pdb',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

len(np.unique(u.atoms.resids))

In [ ]:
visu(u)

In [ ]:

inner_layers,outer_layers,res_list = surface_comp_new(u)


In [ ]:
all_list = ' '
for item in res_list:
    all_list += str(item) + ' '



inner_layer_list = ' '
for item in inner_layers:
    inner_layer_list += str(item) + ' '
    
    
outer_layer_list = ' '
for item in outer_layers:
    outer_layer_list += str(item) + ' '

    
selection = 'resindex '
for item in res_list:
        selection += str(item)+ ' or resindex '
selection = selection[0:-13]
    

dist_water_inner = 3.3
dist_water_outer = 3.0

#u.trajectory[100000]

# Step 2: Water within dist_water of the CORE (clustered) residues

select_inner = f" name OW  and (around {dist_water_inner} (protein and resid {inner_layer_list}))"

select_outer = f" name OW  and (around {dist_water_outer} (protein and resid {outer_layer_list}))"

water_inner = u.select_atoms(select_inner)

water_outer = u.select_atoms(select_outer)

water_inner_resids = water_inner.indices
water_outer_resids = water_outer.indices

common_resids = set(water_inner_resids).intersection(set(water_outer_resids))

water_inner_resids = set(water_inner_resids)-common_resids
    
idx_sel = "@" + ",".join(str(r) for r in water_inner_resids)
idx_sel_2 = "@" + ",".join(str(r) for r in water_outer_resids)

In [ ]:

len(set(water_inner_resids)), len( set(water_outer_resids)), len( set(common_resids)),
#len(inner_layers),len(outer_layers),len(res_list)


In [ ]:

#visul10(u,res_list,outer_layer_list,idx_sel)
view = visul10(u,inner_layer_list,outer_layer_list,{})
view


In [ ]:

view.render_image(factor=10,
                  transparent=True,
                 trim=True)
#view.download_image(filename='output.png')

In [ ]:

# --- Compute center of condensate ---
com = u.select_atoms(f"{selection}").atoms.center_of_mass()

selection = u.select_atoms(f"(prop z > {com[2]})")

# Write cross section
selection.write(f"cross_section.pdb")

print("Number of atoms in slice:", len(selection))

In [ ]:
u = mda.Universe(f'cross_section.pdb',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

In [ ]:

dist_water_inner = 3.3
dist_water_outer = 3.0

select_inner = f" name OW HW1 HW2  and (around {dist_water_inner} (protein and resid {inner_layer_list}))"

select_outer = f" name OW HW1 HW2  and (around {dist_water_outer} (protein and resid {outer_layer_list}))"

water_inner = u.select_atoms(select_inner)

water_outer = u.select_atoms(select_outer)

water_inner_resids = water_inner.indices
water_outer_resids = water_outer.indices

common_resids = set(water_inner_resids).intersection(set(water_outer_resids))

water_inner_resids = set(water_inner_resids)-common_resids
    
idx_sel = "@" + ",".join(str(r) for r in water_inner_resids)
idx_sel_2 = "@" + ",".join(str(r) for r in water_outer_resids)

In [ ]:

visul10(u,inner_layer_list,outer_layer_list,idx_sel)



In [ ]:

view = visul13(u,inner_layer_list,outer_layer_list,idx_sel,idx_sel_2)
view

In [ ]:

view = visul14(u,inner_layer_list,outer_layer_list,idx_sel,idx_sel_2)
view

In [ ]:
view = visul10_water_new2(u,inner_layer_list,outer_layer_list,idx_sel,idx_sel_2)
view

In [ ]:
view = visul10_water_new(u,inner_layer_list,outer_layer_list,"986 4982 5910 8352 9397")
view

In [ ]:

view.render_image(factor=5,
                  transparent=True,
                 trim=True)


In [ ]:

def visul14(unive,s1,s2,water1,water2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
#     view.add_representation('surface',sele=f"(nucleic) and ( {s1})",
#                             opacity=0.5,aspectRatio=2,color='red')  
    
    
#     view.add_representation('surface',sele=f"(protein) and ( {s1})",
#                             opacity=0.5,aspectRatio=4,color='yellow')  
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=0.4,aspectRatio=5,
                            color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(protein) and ({s2})",
                            opacity=0.4,
                            aspectRatio=5,
                            color='green')

    view.add_representation('ball+stick',sele=f"{water1}",
                            opacity=1.5,
                            aspectRatio=6)

    
    view.add_representation('ball+stick',sele=f"{water2}",
                            opacity=1.5,
                            aspectRatio=6)
        

    #view.add_unitcell(color='white')
    return view

In [ ]:

def visul10_water_new(unive,s1,s2,water):
    
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=0.5,aspectRatio=2,color='red'
                           )  
    
    
    view.add_representation('surface',sele=f"(protein) and ( {s1})",
                            opacity=0.1,
                            aspectRatio=3,
                            color='yellow'
                           )  
    

    view.add_representation('surface',sele=f"(protein) and ({s2})",
                            opacity=0.1,
                            aspectRatio=3,
                            color='green'
                           )

    view.add_representation('ball+stick',sele=f" SOL and ({water})",
                            opacity=1.5,
                            aspectRatio=3,
                            #color='blue'
                           )
        


#    view.add_unitcell(color='white')
    return view


def visul10_water_new2(unive,s1,s2,water1,water2):
    
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=0.5,aspectRatio=2,color='red'
                           )  
    
    
    view.add_representation('surface',sele=f"(protein) and ( {s1})",
                            opacity=0.05,
                            aspectRatio=3,
                            color='yellow'
                           )  
    

    view.add_representation('surface',sele=f"(protein) and ({s2})",
                            opacity=0.05,
                            aspectRatio=3,
                            color='green'
                           )

    view.add_representation('ball+stick',sele=f" SOL ",
                            opacity=1.5,
                            aspectRatio=5,
                            #color='blue'
                           )
    view.add_representation('ball+stick',sele=f"{water1}",
                            opacity=1.5,
                            aspectRatio=6,
                           color='red')

    
    view.add_representation('ball+stick',sele=f"{water2}",
                            opacity=1.5,
                            aspectRatio=6,
                           color='blue')
        


#    view.add_unitcell(color='white')
    return view

## picking specifc atoms for HB atoms 

In [ ]:

def visul10_water(unive,s1,s2,water,water2):
    
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=0.5,aspectRatio=2,color='red'
                           )  
    
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=0.5,
                            aspectRatio=3,
                            color='yellow'
                           )  
    

    view.add_representation('ball+stick',sele=f"(protein) and ({s2})",
                            opacity=0.5,
                            aspectRatio=5,
                            color='green'
                           )

    view.add_representation('ball+stick',sele=f" SOL and ({water})",
                            opacity=1.5,
                            aspectRatio=5,
                            #color='blue'
                           )
        
    view.add_representation('ball+stick',sele=f" SOL and ({water2})",
                            opacity=1.5,
                            aspectRatio=5,
                            #color='blue'
                           )

#    view.add_unitcell(color='white')
    return view


In [ ]:

view = visul10_water(u,inner_layer_list,outer_layer_list,idx_sel,idx_sel_2)
view


In [ ]:
view.render_image(factor=5,
                  transparent=True,
                 trim=True)

In [ ]:
view = visu2(u)
view

In [ ]:

view.render_image(factor=10,
                  transparent=True,
                 trim=True)


In [ ]:
def visu2(u):
    view=nv.show_mdanalysis(u)
    view.clear_representations()
    view.add_representation('ball+stick',sele='protein and ( 631 630 632  741 742 743)',
                            aspectRatio=2.9,opacity=0.5,color='grey'
                           )    
    view.add_representation('ball+stick',sele='protein and (631 742 743)',aspectRatio=3)

                              
    #view.add_representation('surface',sele='protein',opacity=0.3,color='silver')

   # view.add_representation('surface',sele='LYS',opacity=0.8,color='green')
    #view.add_representation('ball+stick',sele='LYS',aspectRatio=2)
    #view.add_representation('ball+stick',sele='ions',aspectRatio=3)
    view.add_representation('ball+stick',sele='SOL and (8215)',aspectRatio=2)
    #view.add_unitcell(color='white')
    return view

def visu2(u):
    view=nv.show_mdanalysis(u)
    view.clear_representations()
    view.add_representation('ball+stick',sele='protein and ( 646 647 648  741 742 743)',
                            aspectRatio=2.9,opacity=0.5,color='grey'
                           )    
    view.add_representation('ball+stick',sele='protein and (646 742)',aspectRatio=3)

                              
    #view.add_representation('surface',sele='protein',opacity=0.3,color='silver')

   # view.add_representation('surface',sele='LYS',opacity=0.8,color='green')
    #view.add_representation('ball+stick',sele='LYS',aspectRatio=2)
    #view.add_representation('ball+stick',sele='ions',aspectRatio=3)
    view.add_representation('ball+stick',sele='SOL and (2987 4024 5336 5982 )',aspectRatio=2)
    #view.add_unitcell(color='white')
    return view


In [ ]:
u = u2

for ts in u.trajectory:

    centroid, to_cluster, clustered_list, res_list = clustering(
        u, ts, eps=9
    )
         
    selection = 'resindex '
    for item in res_list:
            selection += str(item)+ ' or resindex '
    selection = selection[0:-13]

    res_list = np.array(res_list)

    selection_2 = ' '
    for item in res_list:
            selection_2 += str(item)+ ' '       

    sel = np.unique(univ.select_atoms(f"(nucleic or protein) and not ({selection})").resids)

    selection_3 = ' '
    for item in sel:
            selection_3 += str(item)+ ' '


In [ ]:

dist_water = 3.0
#u.trajectory[67000]
# Step 2: Water within dist_water of the CORE (clustered) residues
select = f" name OW HW1 HW2 and (around {dist_water} (protein and resid {selection_2}))"
water_near_core = u.select_atoms(select)

water_resids = water_near_core.indices

idx_sel = "@" + ",".join(str(r) for r in water_resids)


In [ ]:

view = visul6_(u,selection_2,idx_sel)
view


In [ ]:
com

In [ ]:
com = u.select_atoms(f"{selection}").atoms.center_of_mass()

In [ ]:

view.render_image(factor=10,
                  transparent=True,
                 trim=True)
